In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

import scripts.stock_plots as stock_plots
from scripts.preparation import download_data

# Top trendings analysis settings


In [2]:
def extract_ticker(df_row, russell_list):
    capital = 0
    for letter in df_row:
        if letter.isupper():
            capital += 1
        else:
            break

    
    answer = df_row[:capital-1]
    debug = answer
    white_list = ["SMCI", "MSTR", "NVAX"]

    while len(answer) > 0:
        if answer in (russell_list + white_list):
            return answer
        else:
            answer = answer[:len(answer)-1]
    
    print("Can't find: ", debug)
    return None

In [3]:
russell_table = pd.read_html("https://en.wikipedia.org/wiki/Russell_1000_Index")
russell_list = list(russell_table[2]["Ticker"])


top100 = pd.read_html("https://www.tradingview.com/markets/stocks-usa/market-movers-active/")
ticker_list = list(top100[0]["Symbol"].apply(extract_ticker, russell_list=russell_list))
ticker_list = [i for i in ticker_list if i is not None]

Can't find:  ENVX


In [4]:
ticker_object = download_data(ticker_list)
print_all = False

In [5]:
# holding = "SHOP PINS ABNB"
# holding_list = holding.split()
# # holding_list = ["DHR", "MRK", "ORCL", "PG", "CSCO", "COST", "ADBE", "AAPL", "META", "VRT"]

# ticker_list = list(set(holding_list))

# ticker_object = download_data(ticker_list)
# print_all = True

# Start Analysis

In [6]:
saved = 0
strangle_plot = {}
trend_plot = {}
for ticker in ticker_list:
    stock_plot = stock_plots.PlotInfo(ticker_object.tickers[ticker], ticker, "100d")
    # candle = stock_plot.generate_quick_analysis()
    candle = stock_plot.generate_candle_plot_no_op()
    # candle = stock_plot.generate_candle_plot()

    stock_plot.df["day_chg"] = abs(stock_plot.df["Close"] - stock_plot.df["Open"])
    stock_plot.df["day_chg_10ma"] =  stock_plot.df["day_chg"].rolling(10).mean()
    huge_day_chg = stock_plot.df["day_chg"].iloc[-1] > (2 * stock_plot.df["day_chg_10ma"].iloc[-1])
    
    if print_all:
        print(ticker, saved)
        candle.show()
        saved += 1

    else:
        if huge_day_chg:
            strangle_plot[ticker] = candle
            saved += 1
        

        elif ((sum(stock_plot.macd_analysis["macd_up_idx"].tail(2)) > 0 
               or sum(stock_plot.macd_analysis["bull_idx"].tail(2)) > 0)
               and sum(stock_plot.macd_analysis["macd_down_idx"].tail(2)) == 0):
            trend_plot[ticker] = candle
            saved += 1
        
    if saved > 35:
        break

print(saved)



36


## strangle

In [7]:
for i, (key, candle_plot) in enumerate(strangle_plot.items()):
    print(key, f"{i+1}/{len(strangle_plot)}", "strangle")
    candle_plot.show()


META 1/10 strangle


GOOG 2/10 strangle


CCL 3/10 strangle


HD 4/10 strangle


ALNY 5/10 strangle


ISRG 6/10 strangle


POOL 7/10 strangle


LOW 8/10 strangle


TSCO 9/10 strangle


GM 10/10 strangle


## trend

In [8]:
for i, (key, candle_plot) in enumerate(trend_plot.items()):
    print(key, f"{i+1}/{len(trend_plot)}", "trend_plot")
    candle_plot.show()

TSLA

 1/26 trend_plot


AAPL 2/26 trend_plot


LLY 3/26 trend_plot


V 4/26 trend_plot


...

M 22/26 trend_plot


NKE 23/26 trend_plot


NOW 24/26 trend_plot


VZ 25/26 trend_plot


SLB 26/26 trend_plot


# End analysis